In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import duckdb

In [2]:
df = pd.read_csv('df_features.csv')
df1 = pd.read_csv('df_label_window.csv')

In [3]:
df.duplicated().sum()

np.int64(0)

In [4]:
df.head()

,Invoice,StockCode,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Purchased_Date
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12-01
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01
3,489434,22041,48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12-01
4,489434,21232,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12-01


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 530514 entries, 0 to 530513
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Invoice         530514 non-null  int64  
 1   StockCode       530514 non-null  str    
 2   Quantity        530514 non-null  int64  
 3   InvoiceDate     530514 non-null  str    
 4   Price           530514 non-null  float64
 5   Customer ID     530514 non-null  float64
 6   Country         530514 non-null  str    
 7   Revenue         530514 non-null  float64
 8   Purchased_Date  530514 non-null  str    
dtypes: float64(3), int64(2), str(4)
memory usage: 36.4 MB


In [6]:
df['InvoiceDate']=pd.to_datetime(df.InvoiceDate)

In [7]:
df['Purchased_Date']=pd.to_datetime(df.Purchased_Date)

In [8]:
df['Customer ID']=df['Customer ID'].astype(int)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 530514 entries, 0 to 530513
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Invoice         530514 non-null  int64         
 1   StockCode       530514 non-null  str           
 2   Quantity        530514 non-null  int64         
 3   InvoiceDate     530514 non-null  datetime64[us]
 4   Price           530514 non-null  float64       
 5   Customer ID     530514 non-null  int64         
 6   Country         530514 non-null  str           
 7   Revenue         530514 non-null  float64       
 8   Purchased_Date  530514 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(2), int64(3), str(2)
memory usage: 36.4 MB


## Recency

In [10]:
df.head(3)

,Invoice,StockCode,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Purchased_Date
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12-01
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01


In [11]:
df

,Invoice,StockCode,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Purchased_Date
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,2009-12-01
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009-12-01
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009-12-01
3,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,2009-12-01
4,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,2009-12-01
...,...,...,...,...,...,...,...,...,...
530509,555147,84707A,2,2011-05-31 15:41:00,9.95,13505,Switzerland,19.90,2011-05-31
530510,555147,84711A,2,2011-05-31 15:41:00,9.95,13505,Switzerland,19.90,2011-05-31
530511,555147,84711B,2,2011-05-31 15:41:00,9.95,13505,Switzerland,19.90,2011-05-31
530512,555147,23101,12,2011-05-31 15:41:00,0.83,13505,Switzerland,9.96,2011-05-31


In [12]:
df.groupby(by='Customer ID')['Purchased_Date'].max()

Customer ID
12346   2011-01-18
12347   2011-04-07
12348   2011-04-05
12349   2010-10-28
12350   2011-02-02
           ...    
18283   2011-05-23
18284   2010-10-04
18285   2010-02-17
18286   2010-08-20
18287   2011-05-22
Name: Purchased_Date, Length: 4935, dtype: datetime64[us]

In [13]:
Recency=pd.DataFrame( pd.Timestamp('2011-06-01') - df.groupby(by='Customer ID')['Purchased_Date'].max())
Recency.columns=['Recency']
Recency

,Recency
Customer ID,
12346,134 days
12347,55 days
12348,57 days
12349,216 days
12350,119 days
...,...
18283,9 days
18284,240 days
18285,469 days


In [14]:
Frequency=pd.DataFrame(df.groupby('Customer ID')['Purchased_Date'].nunique())
Frequency.columns=['Frequency']
Frequency

,Frequency
Customer ID,
12346,8
12347,4
12348,4
12349,3
12350,1
...,...
18283,10
18284,1
18285,1


In [15]:
Monetary=pd.DataFrame(df.groupby('Customer ID')['Revenue'].sum())
Monetary.columns=['Monetary']
Monetary

,Monetary
Customer ID,
12346,77556.46
12347,2434.96
12348,1709.40
12349,2671.14
12350,334.40
...,...
18283,1134.89
18284,461.68
18285,427.00


In [16]:
Tenure=pd.DataFrame(pd.Timestamp('2011-06-01') - df.groupby(by='Customer ID')['Purchased_Date'].min())
Tenure.columns=['Tenure']
Tenure

,Tenure
Customer ID,
12346,534 days
12347,213 days
12348,247 days
12349,398 days
12350,119 days
...,...
18283,467 days
18284,240 days
18285,469 days


In [17]:
AvgOrderValue=Monetary['Monetary']/Frequency['Frequency']
AvgOrderValue=pd.DataFrame(AvgOrderValue)
AvgOrderValue.columns=['AvgOrderValue']
AvgOrderValue

,AvgOrderValue
Customer ID,
12346,9694.5575
12347,608.7400
12348,427.3500
12349,890.3800
12350,334.4000
...,...
18283,113.4890
18284,461.6800
18285,427.0000


In [18]:
df.groupby(by='Customer ID')['StockCode'].nunique()

Customer ID
12346     27
12347     90
12348     25
12349     90
12350     17
        ... 
18283    240
18284     28
18285     12
18286     67
18287     93
Name: StockCode, Length: 4935, dtype: int64

In [19]:
ProductDiversity=pd.DataFrame(df.groupby(by='Customer ID')['StockCode'].nunique())
ProductDiversity.columns=['ProductDiversity']
ProductDiversity

,ProductDiversity
Customer ID,
12346,27
12347,90
12348,25
12349,90
12350,17
...,...
18283,240
18284,28
18285,12


In [20]:
df['Purchased_Date']

0        2009-12-01
1        2009-12-01
2        2009-12-01
3        2009-12-01
4        2009-12-01
            ...    
530509   2011-05-31
530510   2011-05-31
530511   2011-05-31
530512   2011-05-31
530513   2011-05-31
Name: Purchased_Date, Length: 530514, dtype: datetime64[us]

In [21]:
df['is_weekend']= df['Purchased_Date'].dt.dayofweek>=5

In [22]:
WeekendRatio=pd.DataFrame(df.groupby(by='Customer ID')['is_weekend'].mean())
WeekendRatio.columns=['WeekendRatio']
WeekendRatio

,WeekendRatio
Customer ID,
12346,0.000000
12347,0.322581
12348,0.000000
12349,0.000000
12350,0.000000
...,...
18283,0.225513
18284,0.000000
18285,0.000000


In [23]:
customer_features=pd.merge(Recency,Frequency,how='inner',on='Customer ID')

In [24]:
customer_features=pd.merge(customer_features,Monetary,how='inner',on='Customer ID')

In [25]:
customer_features=pd.merge(customer_features,Tenure,how='inner',on='Customer ID')

In [26]:
customer_features=pd.merge(customer_features,AvgOrderValue,how='inner',on='Customer ID')

In [27]:
customer_features=pd.merge(customer_features,ProductDiversity,how='inner',on='Customer ID')

In [28]:
customer_features=pd.merge(customer_features,WeekendRatio,how='inner',on='Customer ID')

In [29]:
customer_features

,Recency,Frequency,Monetary,Tenure,AvgOrderValue,ProductDiversity,WeekendRatio
Customer ID,,,,,,,
12346,134 days,8,77556.46,534 days,9694.5575,27,0.000000
12347,55 days,4,2434.96,213 days,608.7400,90,0.322581
12348,57 days,4,1709.40,247 days,427.3500,25,0.000000
12349,216 days,3,2671.14,398 days,890.3800,90,0.000000
12350,119 days,1,334.40,119 days,334.4000,17,0.000000
...,...,...,...,...,...,...,...
18283,9 days,10,1134.89,467 days,113.4890,240,0.225513
18284,240 days,1,461.68,240 days,461.6800,28,0.000000
18285,469 days,1,427.00,469 days,427.0000,12,0.000000


In [30]:
df1['InvoiceDate']=pd.to_datetime(df1.InvoiceDate)
df1['Purchased_Date']=pd.to_datetime(df1.Purchased_Date)

In [31]:
df1['Customer ID']=df1['Customer ID'].astype(int)

In [32]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 248979 entries, 0 to 248978
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Invoice         248979 non-null  int64         
 1   StockCode       248979 non-null  str           
 2   Quantity        248979 non-null  int64         
 3   InvoiceDate     248979 non-null  datetime64[us]
 4   Price           248979 non-null  float64       
 5   Customer ID     248979 non-null  int64         
 6   Country         248979 non-null  str           
 7   Revenue         248979 non-null  float64       
 8   Purchased_Date  248979 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(2), int64(3), str(2)
memory usage: 17.1 MB


In [33]:
labels = df1.groupby('Customer ID')['Revenue'].sum()
labels = labels.reset_index()
labels.columns = ['Customer ID', 'future_6m_clv']

In [34]:
labels.shape

(3548, 2)

In [35]:
customer_features.shape

(4935, 7)

In [36]:
final = customer_features.merge(labels, on='Customer ID', how='left')
final['future_6m_clv'] = final['future_6m_clv'].fillna(0)

In [37]:
final

,Customer ID,Recency,Frequency,Monetary,Tenure,AvgOrderValue,ProductDiversity,WeekendRatio,future_6m_clv
0,12346,134 days,8,77556.46,534 days,9694.5575,27,0.000000,0.00
1,12347,55 days,4,2434.96,213 days,608.7400,90,0.322581,2486.57
2,12348,57 days,4,1709.40,247 days,427.3500,25,0.000000,310.00
3,12349,216 days,3,2671.14,398 days,890.3800,90,0.000000,1757.55
4,12350,119 days,1,334.40,119 days,334.4000,17,0.000000,0.00
...,...,...,...,...,...,...,...,...,...
4930,18283,9 days,10,1134.89,467 days,113.4890,240,0.225513,1530.01
4931,18284,240 days,1,461.68,240 days,461.6800,28,0.000000,0.00
4932,18285,469 days,1,427.00,469 days,427.0000,12,0.000000,0.00
4933,18286,285 days,2,1296.43,532 days,648.2150,67,0.000000,0.00


In [38]:
print(final.shape)                          
print((final['future_6m_clv'] == 0).sum())   

(4935, 9)
2333


In [39]:
final['Recency']=final['Recency'].dt.days
final['Tenure']=final['Tenure'].dt.days

In [40]:
final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4935 entries, 0 to 4934
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Customer ID       4935 non-null   int64  
 1   Recency           4935 non-null   int64  
 2   Frequency         4935 non-null   int64  
 3   Monetary          4935 non-null   float64
 4   Tenure            4935 non-null   int64  
 5   AvgOrderValue     4935 non-null   float64
 6   ProductDiversity  4935 non-null   int64  
 7   WeekendRatio      4935 non-null   float64
 8   future_6m_clv     4935 non-null   float64
dtypes: float64(4), int64(5)
memory usage: 347.1 KB


In [46]:
final.to_csv('C:\\Users\\NOTHING\\Pictures\\ValueLens AI\\data\\processed\\customer_features.csv', index=False)

In [42]:
# | Feature              | Formula                                    | Unit        |
# | -------------------- | ------------------------------------------ | ----------- |
# | **Recency**          | `(cutoff_date - last_purchase_date).days`  | **Days**    |
# | **Tenure**           | `(cutoff_date - first_purchase_date).days` | **Days**    |
# | **Frequency**        | Number of distinct invoices                | Count       |
# | **Monetary**         | Sum of Revenue                             | Currency    |
# | **AvgOrderValue**    | `Monetary / Frequency`                     | Currency    |
# | **ProductDiversity** | Number of distinct StockCodes              | Count       |
# | **WeekendRatio**     | Weekend invoices / Total invoices          | Ratio (0–1) |
# | **Future_6m_CLV**    | Sum of Revenue in label window             | Currency    |